In [14]:

from Training_utils import train_loader, train
import torch_geometric as tg
import torch
from SUPG_prediction_models import *

batch_size = 15
train_loader = train_loader(batch_size=batch_size)

class att(torch.nn.Module):
    def __init__(self):
        super().__init__()


        self.pattn1 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        self.pattn2 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        self.pattn3 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        self.pattn4 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        self.pattn5 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        self.pattn6 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        self.pattn7 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        self.pattn8 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        self.pattn9 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        self.pattn10 = tg.nn.attention.PerformerAttention(channels=10, heads=1)
        
        self.mlp = tg.nn.models.MLP(
            in_channels=10,
            hidden_channels=5,
            num_layers=3,
            out_channels=1,
        )

    def forward(self, data) -> torch.Tensor:
        batch, x  = data.batch, data.x
        h, mask = utils.to_dense_batch(x=x, batch=batch)
        h = self.pattn1(h).relu()
        h = self.pattn2(h).relu()
        h = self.pattn3(h).relu()
        h = self.pattn4(h).relu()
        h = self.pattn5(h).relu()
        h = self.pattn6(h).relu()
        h = self.pattn7(h).relu()
        h = self.pattn8(h).relu()
        h = self.pattn9(h).relu()
        h = self.pattn10(h)[mask]
        h = self.mlp(h)

        return h
    

class lmlp(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = tg.nn.models.MLP(
            in_channels=10,
            hidden_channels=32,
            num_layers=50,
            out_channels=1,
            dropout=0.2
        )

    def forward(self, data) -> torch.Tensor:
        x = data.x
        h = self.mlp(x)

        return h
    
model = lmlp()

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, factor=0.9, patience=50)

In [2]:
curr_loss = 0.026

In [10]:
optimizer = torch.optim.SGD(model.parameters(), lr=3e-4, momentum=0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer=optimizer, T_0=100)

In [15]:

for i in range(10000):
    loss = train(model=model, loader=train_loader, optimizer=optimizer, device='cpu')
    if loss > curr_loss:
        print(f"iteration: {i}, loss: {loss}, model loss: {curr_loss}:")
    elif loss <= curr_loss:
        curr_loss = loss
        print(f"iteration: {i}, new model loss: {loss}")
        torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'loss': loss}, "data/models/llmlp_supervised.pth")

    #scheduler.step(loss)

iteration: 0, loss: 0.9461534917354584, model loss: 0.026:
iteration: 1, loss: 0.9461601873238882, model loss: 0.026:
iteration: 2, loss: 0.8246961832046509, model loss: 0.026:
iteration: 3, loss: 0.8760413030783335, model loss: 0.026:
iteration: 4, loss: 0.8633649150530497, model loss: 0.026:
iteration: 5, loss: 0.8041434039672216, model loss: 0.026:
iteration: 6, loss: 0.844035396973292, model loss: 0.026:
iteration: 7, loss: 0.7809330324331919, model loss: 0.026:
iteration: 8, loss: 0.7482530375321707, model loss: 0.026:
iteration: 9, loss: 0.8334689935048422, model loss: 0.026:
iteration: 10, loss: 0.7318139423926672, model loss: 0.026:
iteration: 11, loss: 0.7223115513722101, model loss: 0.026:
iteration: 12, loss: 0.7759598592917124, model loss: 0.026:
iteration: 13, loss: 0.7100467383861542, model loss: 0.026:
iteration: 14, loss: 0.7115270594755808, model loss: 0.026:
iteration: 15, loss: 0.7284322927395502, model loss: 0.026:
iteration: 16, loss: 0.6291764105359713, model loss

In [25]:
import torch
crow_indices = [0, 1, 3]
col_indices = [ 1, 0, 1]
values = [2, 3, 4]
A = torch.sparse_csr_tensor(torch.tensor(crow_indices, dtype=torch.int64),
                        torch.tensor(col_indices, dtype=torch.int64),
                        torch.tensor(values), dtype=torch.double)

A[0,1]

tensor(2., dtype=torch.float64)